# 🚦 Traffic Signal Classifier Training

Trains a whole-frame image classifier (Green vs Red) instead of an object
detector - no bounding boxes, the model just looks at an entire camera frame
and says which signal color it is. Architecture matches `torchnn.py` exactly
(a plain CNN trained from scratch, no pretrained weights), so the state_dict
this notebook produces is a drop-in match for `webapp/detector.py`.

**Workflow**
1. Check environment (no install needed - Colab ships with PyTorch/torchvision)
2. Upload and prepare the dataset zip (`data.zip`, with `Green/` and `Red/` subfolders)
3. Sanity-check the data
4. Define the model architecture (identical to `torchnn.py`)
5. Train (with augmentation and early stopping - see the caveat below)
6. Evaluate
7. Preview predictions
8. Download the trained model

**Before you start:** `Runtime > Change runtime type > T4 GPU` speeds this up, but
isn't required - the dataset is tiny enough that CPU works fine too.

> **Two things worth knowing going in.** First, this dataset is only 70 images
> (34 Green / 36 Red), and this architecture has no pretrained weights at all
> (unlike the earlier YOLOv8 notebook, which started from COCO-pretrained
> weights) - training on all 70 images with no augmentation makes this network
> memorize the training set almost immediately (confirmed while building this:
> loss hit 0.0000 by epoch 2), which says nothing about real-world accuracy.
> This notebook adds geometric augmentation and validation-based early stopping
> to push back against that, but treat the reported accuracy as optimistic, not
> a guarantee.
> Second, there's no detection step anymore - the whole frame is classified
> directly, so how the ESP32-CAM is framed matters a lot. This works best when
> the signal light fills a meaningful portion of the frame, not a tiny speck in
> a wide scene.

## 1. Environment check

No `pip install` needed here - Colab already has PyTorch, torchvision, and
scikit-learn preinstalled.

In [ ]:
import torch
import torchvision

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("GPU available:", torch.cuda.is_available())

## 2. Upload and prepare the dataset

Run the next cell, then click **Choose Files** and select `data.zip` - the one
with `Green/` and `Red/` subfolders inside (regenerate it from your sorted
`data/` folder if you're not sure it's current).

In [ ]:
from google.colab import files

print("Select your dataset zip file (e.g. 'data.zip')")
uploaded = files.upload()
if len(uploaded) != 1:
    print(f"Warning: expected 1 file, got {len(uploaded)}. Using the first one.")
zip_name = next(iter(uploaded))
print(f"\nUploaded '{zip_name}' ({len(uploaded[zip_name]) / 1e6:.1f} MB)")

In [ ]:
import shutil
import zipfile
from pathlib import Path

EXTRACT_DIR = Path("/content/dataset")
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

with zipfile.ZipFile(f"/content/{zip_name}", "r") as zf:
    zf.extractall(EXTRACT_DIR)

# The zip might extract as data/Green,Red or flatten straight to Green,Red -
# find whichever directory directly contains >=2 non-empty class subfolders.
candidates = []
for p in [EXTRACT_DIR] + [d for d in EXTRACT_DIR.rglob("*") if d.is_dir()]:
    subdirs = [d for d in p.iterdir() if d.is_dir()]
    if len(subdirs) >= 2 and all(any(d.iterdir()) for d in subdirs):
        candidates.append(p)

assert candidates, f"Could not find class subfolders inside {zip_name} - check it has Green/ and Red/ folders."
DATA_DIR = min(candidates, key=lambda p: len(str(p)))
print("Dataset root:", DATA_DIR)
print("Classes found:", sorted(d.name for d in DATA_DIR.iterdir() if d.is_dir()))

In [ ]:
for d in sorted(DATA_DIR.iterdir()):
    if d.is_dir():
        n = len(list(d.iterdir()))
        print(f"{d.name:10s} {n} images")

### Preview a few samples per class

In [ ]:
import random

import matplotlib.pyplot as plt
from PIL import Image

%matplotlib inline

class_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])
fig, axes = plt.subplots(len(class_dirs), 4, figsize=(14, 3.2 * len(class_dirs)))
for row, class_dir in enumerate(class_dirs):
    images = list(class_dir.iterdir())
    sample = random.sample(images, min(4, len(images)))
    for col in range(4):
        ax = axes[row][col]
        if col < len(sample):
            ax.imshow(Image.open(sample[col]).convert("RGB"))
            ax.set_title(class_dir.name, fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Model architecture

Identical to the `ImageClassifier` in `torchnn.py` and in `webapp/detector.py`
- don't change the layer shapes here without updating both of those to match,
or a trained state_dict won't load.

In [ ]:
from torch import nn


class ImageClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = nn.Sequential(
            # Input: 3 channels (RGB), 224x224
            nn.Conv2d(3, 32, (3, 3)),   # Output: 32x222x222
            nn.ReLU(),
            nn.MaxPool2d(2),            # Output: 32x111x111
            nn.Conv2d(32, 64, (3, 3)),  # Output: 64x109x109
            nn.ReLU(),
            nn.MaxPool2d(2),            # Output: 64x54x54
            nn.Conv2d(64, 128, (3, 3)), # Output: 128x52x52
            nn.ReLU(),
            nn.MaxPool2d(2),            # Output: 128x26x26
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128 * 26 * 26, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.model(x)


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Data transforms

Training gets geometric augmentation (flip, rotation, slight translate/zoom)
to squeeze more variety out of 70 images. Deliberately **no color/hue jitter**
- this task is entirely about telling red from green apart, so anything that
shifts color would blur the exact signal the model needs to learn. Validation
uses a clean, non-augmented transform so its accuracy reflects real performance.

In [ ]:
from torchvision import datasets, transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

## 4. Train

Two `ImageFolder` instances (one per transform) sharing the same random
train/val split indices, so augmentation only ever touches the training half.
The best checkpoint (highest validation accuracy) is saved during training,
not just whatever the last epoch happens to produce, and training stops early
if validation accuracy hasn't improved in `PATIENCE` epochs.

In [ ]:
import json

from torch.optim import Adam
from torch.utils.data import DataLoader, Subset

MODEL_DIR = Path("/content/models")
MODEL_PATH = MODEL_DIR / "signal_classifier.pt"
CLASSES_PATH = MODEL_DIR / "classes.json"

EPOCHS = 60
BATCH_SIZE = 8       # small on purpose - only ~56 training images after the split
LR = 1e-3
VAL_SPLIT = 0.2
PATIENCE = 15
SEED = 42

train_full = datasets.ImageFolder(root=DATA_DIR, transform=train_transform)
eval_full = datasets.ImageFolder(root=DATA_DIR, transform=eval_transform)
classes = train_full.classes
print(f"Classes: {classes} ({len(train_full)} images)")

n = len(train_full)
val_size = max(1, int(VAL_SPLIT * n))
train_size = n - val_size
generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(n, generator=generator).tolist()
train_idx, val_idx = indices[:train_size], indices[train_size:]

train_ds = Subset(train_full, train_idx)
val_ds = Subset(eval_full, val_idx)
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

device = get_device()
print("Training on:", device)

clf = ImageClassifier(len(classes)).to(device)
opt = Adam(clf.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()

history = {"train_loss": [], "val_acc": []}
best_val_acc = 0.0
epochs_without_improvement = 0

MODEL_DIR.mkdir(exist_ok=True, parents=True)

for epoch in range(EPOCHS):
    clf.train()
    running_loss = 0.0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        yhat = clf(X)
        loss = loss_fn(yhat, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        running_loss += loss.item()

    clf.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            yhat = clf(X)
            correct += (torch.argmax(yhat, dim=1) == y).sum().item()
            total += y.size(0)
    val_acc = correct / total if total else 0.0
    train_loss = running_loss / len(train_loader)
    history["train_loss"].append(train_loss)
    history["val_acc"].append(val_acc)

    improved = val_acc > best_val_acc
    if improved:
        best_val_acc = val_acc
        epochs_without_improvement = 0
        torch.save(clf.state_dict(), MODEL_PATH)
    else:
        epochs_without_improvement += 1

    flag = " *" if improved else ""
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {train_loss:.4f} - val_acc: {val_acc:.4f}{flag}")

    if epochs_without_improvement >= PATIENCE:
        print(f"No improvement for {PATIENCE} epochs, stopping early.")
        break

with open(CLASSES_PATH, "w") as f:
    json.dump(classes, f)

print(f"\nBest val_acc: {best_val_acc:.4f}")
print(f"Model saved to {MODEL_PATH}")
print(f"Classes saved to {CLASSES_PATH}")

In [ ]:
# Reload the best checkpoint (not whatever the last epoch happened to leave
# in memory) so every cell below this one evaluates the actual best model.
clf = ImageClassifier(len(classes)).to(device)
clf.load_state_dict(torch.load(MODEL_PATH, map_location=device))
clf.eval()
print("Reloaded best checkpoint for evaluation.")

## 5. Evaluate

With only ~14 validation images, treat these numbers as directional, not a
statistically solid estimate.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

all_preds, all_labels = [], []
clf.eval()
with torch.no_grad():
    for X, y in val_loader:
        X = X.to(device)
        preds = torch.argmax(clf(X), dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.numpy())

print(classification_report(all_labels, all_preds, target_names=classes))
print("Confusion matrix (rows = actual, cols = predicted):", classes)
print(confusion_matrix(all_labels, all_preds))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"])
axes[0].set_title("Training loss")
axes[0].set_xlabel("epoch")

axes[1].plot(history["val_acc"])
axes[1].set_title("Validation accuracy")
axes[1].set_xlabel("epoch")
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## 6. Preview predictions on validation images

In [ ]:
sample_idx = random.sample(val_idx, min(6, len(val_idx)))
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, idx in zip(axes.ravel(), sample_idx):
    img_path, label_idx = train_full.samples[idx]
    img = Image.open(img_path).convert("RGB")
    tensor = eval_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(clf(tensor), dim=1)[0]
        conf, pred_idx = torch.max(probs, dim=0)
    actual = classes[label_idx]
    predicted = classes[pred_idx.item()]
    tag = "OK" if predicted == actual else "WRONG"
    ax.imshow(img)
    ax.set_title(f"actual={actual} pred={predicted} ({conf.item():.0%}) [{tag}]", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Download the trained model

Downloads `signal_classifier.pt` and `classes.json` - drop both into
`webapp/models/` in the project (same folder, both files, no renaming).

In [ ]:
from google.colab import files

files.download(str(MODEL_PATH))
files.download(str(CLASSES_PATH))

## Done

You now have:
- `signal_classifier.pt` - the trained weights (best validation epoch, not
  necessarily the last one)
- `classes.json` - the class name list the model's output indices map to

Drop both into `webapp/models/` (create the folder if it isn't there yet).
`webapp/detector.py` already expects exactly these two filenames.

To reuse the model later outside the webapp:
```python
import json, torch
from PIL import Image
# ImageClassifier + transform definitions from this notebook / webapp/detector.py

with open("classes.json") as f:
    classes = json.load(f)
clf = ImageClassifier(len(classes))
clf.load_state_dict(torch.load("signal_classifier.pt", map_location="cpu"))
clf.eval()
```